# NYC Building Insights

**Team:** cosmic-spaghetti — Mery & Najiihah  
**Links:** [Live App](https://cosmic-spaghetti.streamlit.app/) · [GitHub Repo](https://github.com/advanced-computing/cosmic-spaghetti)

---
*Built as a final project for Advanced Computing, Spring 2025. Data sourced from NYC Open Data.*

Our project is an interactive Streamlit dashboard which provides interesting insights across buildings in New York City. The dashboard aims to provide a foundation that can be built upon, for policymakers and other third parties as it covers building permits, evictions, complaints, and facade inspections across all five boroughs.

## Why It Matters

New York City is currently experiencing its worst housing affordability crisis, largely due to a severe supply-demand mismatch. The housing market has been unable to keep up with the city's population growth, leading to increased rents and eviction rates.

The city also has a building safety crisis driven by aging infrastructure, construction challenges, and widespread scaffolding across the city.

Our dashboard sits at the intersection of both crises by highlighting data that city agencies already collect and making it visible to anyone who wants to understand what's actually happening across NYC's built environment.

**Who is this useful for?**
- Policymakers (DoB, City of New York, FDNY, etc.)
- Renters, buyers, and sellers
- Housing agents
- Mayor Mamdani!

## Evolution of the Project

While the project initially focused on housing affordability and evictions, feedback from the teaching team pushed us to think beyond this scope. We expanded the project to include DOB Permits, Building Footprints, and DOB Complaints to generate a more holistic understanding of housing and buildings in NYC. 

Throughout the semester, we leveraged tools that helped us build more reliably and efficiently:

| Tool | Purpose |
|------|---------|
| **NYC Open Data APIs** | Live data source for all five datasets |
| **BigQuery** | Cloud data warehouse replacing direct API calls |
| **GitHub Actions** | Daily 6am UTC refresh via automated workflows |
| **Streamlit** | Interactive dashboard frontend |
| **Ruff** | Linting and formatting for clean, consistent code |
| **Pandera** | Data validation before BigQuery loads |
| **pytest** | Unit tests on visualization functions |

## The App
Snapshot of the app:

The dashboard has four pages, each focused on a different lens of NYC's building landscape:

- **Proposal** — an overview of the project, its motivation, and the data sources used.
![Project Proposal](project_proposal.png)

- **Building Overview** — explore permit activity across all five boroughs, including new construction and renovation trends mapped by location.
![Building Overview](building_overview.png)

- **Building Evictions** — eviction trends over time with anomaly detection, flagging months where counts exceeded the historical mean + 1 standard deviation. Useful for spotting outliers like the COVID-era drop and the post-moratorium surge.
![Building Evictions](building_evictions.png)

- **Building Complaints** — breakdown of DOB complaint categories, volumes, and response times by borough and priority level.
![Building Complaints](building_complaints.png)

Each page includes collapsible filters (borough, status, priority) and info panels that explain domain-specific terms, so that the dashboard is useful even without a zoning background.

You can also find a copy of our presentation here: [Cosmic Spaghetti Presentation](cosmic_spaghetti.pdf)

## Technical Highlights

### Data Pipeline

Data flows from five NYC Open Data APIs → ETL scripts → BigQuery (`cosmic_spaghetti` dataset) → Streamlit app. Each dataset has its own load script using a **truncate-and-replace** strategy, since BigQuery's free tier doesn't support DML operations.


### Merging Two Permit Datasets

The permits data doesn't live in one place. DOB NOW (newer system, covering GC/PL/ME permit types) and DOB Permit Issuance (older system, covering NB/DM/A1/A2/A3 job types) have different schemas, date formats, and field names. We wrote separate fetch functions for each and standardized borough names, dates, and coordinates before concatenating them into a single table.

### An Interesting Bug

The building footprints JSON endpoint returned malformed geometry data that caused silent errors downstream. The fix was counterintuitive: switch to the **CSV endpoint** instead. The same data, different format — `pd.read_csv(io.StringIO(response.text))` worked cleanly where JSON failed. A good reminder that format matters as much as source.

### Performance

Querying 200k+ rows from BigQuery on every page load isn't viable. We used `@st.cache_data(ttl=3600)` to cache query results for one hour, keeping the app snappy while still reflecting daily data refreshes.

## Key Takeaways

### What the data told us

- **Manhattan has the most unsafe facade filings** by a wide margin across all boroughs.
- **The Bronx has the highest eviction counts** and relatively little renovation activity to show for it.
- **Most building renovation permits are in Manhattan**, but reinvestment is concentrated where safety issues already are.
- **Queens generates the most building complaints** across all five boroughs.

### What we learned

1. **The Agile Manifesto is not just theory.** Being willing to pivot from scope, to architecture, to data format is what got us to the finish line.

2. **Data engineering requires data understanding.** Five datasets from the same source (NYC Open Data) still came with five different schemas and five different quirks. You can't write one generic ETL and call it done.

3. **SQL + Python + `@st.cache` are powerful together.** Filtering in BigQuery before loading into pandas, then caching in Streamlit, made a slow app into a fast one.

---

## Limitations & Future Work

- We'd like to incorporate **Census income data** to better understand housing affordability patterns, especially in the Bronx.
- We're still looking for a **sub-borough GeoJSON** file to enable neighborhood-level mapping.
- There is a **gap in 2025 new building application data** and we're not sure why!

